In [4]:
# Đọc dữ liệu
df = pd.read_csv("wearables_monitoring_data.csv")
df.head()

,Unnamed: 0,X1,age,gender,height,weight,steps,heart_rate,calories,distance,entropy_heart,entropy_steps,resting_heart,corr_heart_steps,norm_heart,intensity_karvonen,sd_norm_heart,steps_times_distance,device,activity
0,0,1,25,1,156.921647,74.872053,62.986954,68.135806,10.173303,0.092330,6.280164,6.281904,62.584922,-0.903196,7.511803,0.042224,1.282418,5.815576,apple watch,Lying
1,1,2,20,1,182.221336,68.475631,16.316123,56.208551,0.168211,0.025374,6.231803,5.714110,54.682353,0.236376,6.551301,0.029993,15.995793,0.414000,apple watch,Lying
2,2,3,32,1,179.784072,75.098988,1.000000,71.707234,15.310934,0.000440,6.134085,6.229458,54.693268,-1.000000,21.370364,0.135383,0.590429,0.000440,apple watch,Lying
3,3,4,31,0,158.224936,63.317797,687.052829,61.474449,2.279399,0.365399,5.931150,6.137201,62.336502,-1.000000,-10.166910,-0.011700,17.857847,251.048078,apple watch,Lying
4,4,5,23,1,177.145549,95.523413,101.474466,73.488328,11.770796,0.000440,6.243469,6.309696,56.299818,1.000000,14.208968,0.161462,4.113836,0.044649,apple watch,Lying


In [5]:
df['activity'].value_counts()

activity
Lying             2201
Running 7 METs    1779
Running 5 METs    1600
Running 3 METs    1516
Sitting           1484
Self Pace walk    1420
Name: count, dtype: int64

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
import joblib

# 1. Đọc dữ liệu
df = pd.read_csv("wearables_monitoring_data.csv")

# 2. Tính BMI
df["BMI"] = df["weight"] / ((df["height"] / 100) ** 2)

# 3. Làm sạch
df = df[df["steps"] > 10]
df = df[df["heart_rate"] < 200]
df = df[df["calories"] > 0]
df = df.dropna()

# 4. Chọn feature
feature_cols = ["steps", "BMI", "age", "gender", "distance"]
cat_cols = ["activity"]

X = df[feature_cols + cat_cols]
y = df[["heart_rate", "calories"]]

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6. Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), feature_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ],
    remainder="drop"
)

# 7. Tối ưu hóa Random Forest
print("Bắt đầu tối ưu hóa Random Forest...")
rf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", MultiOutputRegressor(RandomForestRegressor(random_state=42)))
])

rf_params = {
    'model__estimator__n_estimators': [100, 200, 300, 500],
    'model__estimator__max_depth': [None, 10, 20, 30],
    'model__estimator__min_samples_split': [2, 5, 10],
    'model__estimator__min_samples_leaf': [1, 2, 4]
}

rf_search = RandomizedSearchCV(rf_pipeline, param_distributions=rf_params, 
                               n_iter=10, cv=3, scoring='neg_mean_absolute_error', 
                               random_state=42, n_jobs=-1, verbose=1)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

# 8. Tối ưu hóa XGBoost
print("Bắt đầu tối ưu hóa XGBoost...")
xgb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", MultiOutputRegressor(XGBRegressor(objective="reg:squarederror", random_state=42, verbosity=0)))
])

xgb_params = {
    'model__estimator__n_estimators': [100, 200, 300, 500],
    'model__estimator__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__estimator__max_depth': [3, 4, 6, 8, 10],
    'model__estimator__subsample': [0.6, 0.8, 1.0],
    'model__estimator__colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(xgb_pipeline, param_distributions=xgb_params, 
                                n_iter=15, cv=3, scoring='neg_mean_absolute_error', 
                                random_state=42, n_jobs=-1, verbose=1)
xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_

# 9. Hàm đánh giá
def evaluate_multi(model, X_eval, y_eval, name="Model"):
    y_pred = pd.DataFrame(model.predict(X_eval), columns=y_eval.columns)
    print(f"\n===== {name} Evaluation =====")
    for col in y_eval.columns:
        mae = mean_absolute_error(y_eval[col], y_pred[col])
        rmse = np.sqrt(mean_squared_error(y_eval[col], y_pred[col]))
        r2 = r2_score(y_eval[col], y_pred[col])
        print(f"{col}: MAE={mae:.3f}, RMSE={rmse:.3f}, R2={r2:.3f}")

evaluate_multi(best_rf, X_test, y_test, "Optimized Random Forest")
evaluate_multi(best_xgb, X_test, y_test, "Optimized XGBoost")

# 10. Hàm dự đoán sample
def predict_heart_calories(steps, age, weight, height, gender, distance, activity):
    BMI = weight / ((height / 100) ** 2)
    sample = pd.DataFrame([{
        "steps": steps,
        "BMI": BMI,
        "age": age,
        "gender": gender,
        "distance": distance,
        "activity": activity
    }])
    rf_pred = best_rf.predict(sample)[0]
    xgb_pred = best_xgb.predict(sample)[0]
    print("RF:", dict(zip(y.columns, rf_pred)))
    print("XGB:", dict(zip(y.columns, xgb_pred)))

# 11. Lưu pipeline
joblib.dump(best_rf, "ai-service/app/models/rf_pipeline.pkl")
joblib.dump(best_xgb, "ai-service/app/models/xgb_pipeline.pkl")


Random Forest CV MAE: 8.303 ± 0.171
XGBoost CV MAE: 9.646 ± 0.197

===== Random Forest Evaluation =====
heart_rate: MAE=11.774, RMSE=18.507, R2=0.607
calories: MAE=4.981, RMSE=10.129, R2=0.694

===== XGBoost Evaluation =====
heart_rate: MAE=13.646, RMSE=19.894, R2=0.546
calories: MAE=6.041, RMSE=10.663, R2=0.660


['ai-service/app/models/xgb_pipeline.pkl']